# CVE Spectral Drift — Period Comparison

This notebook demonstrates **spectral drift** between two CVE embedding corpora:
- **Period A**: CVE-1999 → CVE-2014 (`embs_99_to_14.npy`)
- **Period B**: CVE-2015 → CVE-2025 (`embs_15_to_2025.npy`)

The analysis builds a k-NN graph for each period, computes the **graph Laplacian spectrum**,
and quantifies the drift between the two spectral distributions using:
- Wasserstein-1 distance on eigenvalue distributions
- Frobenius norm between normalised Laplacians
- Visual overlay of spectral densities (KDE)

> **Data scientist note**: the embeddings were produced by a sentence-transformer model
> fine-tuned on NVD descriptions. Each row is one CVE entry; the embedding dimension is 384.


In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.stats import wasserstein_distance
from sklearn.neighbors import kneighbors_graph
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / 'data' / 'cve_embeddings_demo'

K = 10        # neighbours for kNN graph
N_EIGS = 200  # Laplacian eigenvalues to compute (top-k smallest non-zero)
SEED = 42

## 1. Load Embeddings

In [ ]:
embs_A_raw = np.load(DATA / 'embs_99_to_14.npy', allow_pickle=True)
embs_B_raw = np.load(DATA / 'embs_15_to_2025.npy', allow_pickle=True)

# Handle object arrays (list-of-arrays storage from older numpy.save)
def _coerce(arr):
    if arr.dtype == object:
        return np.vstack(arr.tolist()).astype(np.float32)
    return arr.astype(np.float32)

embs_A = _coerce(embs_A_raw)
embs_B = _coerce(embs_B_raw)

print(f'Period A (1999-2014): {embs_A.shape}')
print(f'Period B (2015-2025): {embs_B.shape}')

## 2. Subsample for Tractable Laplacian Computation

Computing a Laplacian on the full corpus (100k+ entries) is expensive.
We draw a **stratified random sample** of 5000 entries per period.
Increase `N_SAMPLE` if you have > 32 GB RAM.

In [ ]:
N_SAMPLE = 5_000
rng = np.random.default_rng(SEED)

idx_A = rng.choice(len(embs_A), size=min(N_SAMPLE, len(embs_A)), replace=False)
idx_B = rng.choice(len(embs_B), size=min(N_SAMPLE, len(embs_B)), replace=False)

X_A = embs_A[idx_A]
X_B = embs_B[idx_B]
print(f'Sample A: {X_A.shape}, Sample B: {X_B.shape}')

## 3. Build Symmetric k-NN Graph & Normalised Graph Laplacian

We use the **symmetric normalised Laplacian**:

$$L_{\text{sym}} = I - D^{-1/2} A D^{-1/2}$$

whose eigenvalues lie in [0, 2]. A spectral shift between periods
indicates that the geometry of the semantic neighbourhood structure has changed.

In [ ]:
def build_normalised_laplacian(X, k=K):
    A = kneighbors_graph(X, n_neighbors=k, mode='connectivity',
                         include_self=False, n_jobs=-1)
    # Symmetrise
    A = (A + A.T).astype(np.float32)
    A.data = np.ones_like(A.data)  # binarise
    d = np.asarray(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
    D_inv_sqrt = sp.diags(d_inv_sqrt)
    L = sp.eye(X.shape[0]) - D_inv_sqrt @ A @ D_inv_sqrt
    return L.tocsr()

print('Building Laplacian A ...')
L_A = build_normalised_laplacian(X_A)
print('Building Laplacian B ...')
L_B = build_normalised_laplacian(X_B)
print('Done.')

## 4. Compute Eigenvalue Spectra

In [ ]:
def compute_spectrum(L, k=N_EIGS):
    k = min(k, L.shape[0] - 2)
    eigs, _ = spla.eigsh(L, k=k, which='SM', tol=1e-4, maxiter=3000)
    return np.sort(np.real(eigs))

print('Computing spectrum A ...')
eigs_A = compute_spectrum(L_A)
print('Computing spectrum B ...')
eigs_B = compute_spectrum(L_B)
print(f'Spectrum A: min={eigs_A.min():.4f}, max={eigs_A.max():.4f}')
print(f'Spectrum B: min={eigs_B.min():.4f}, max={eigs_B.max():.4f}')

## 5. Drift Metrics

In [ ]:
# Wasserstein-1 distance between eigenvalue distributions
w1 = wasserstein_distance(eigs_A, eigs_B)

# Mean spectral gap (smallest non-trivial eigenvalue — proxy for cluster separation)
gap_A = eigs_A[eigs_A > 1e-6][0] if np.any(eigs_A > 1e-6) else 0.0
gap_B = eigs_B[eigs_B > 1e-6][0] if np.any(eigs_B > 1e-6) else 0.0

# Mean eigenvalue shift
mean_shift = np.mean(eigs_B) - np.mean(eigs_A)

print('=== Spectral Drift Metrics ===')
print(f'Wasserstein-1 distance:  {w1:.6f}')
print(f'Spectral gap A (1999-2014): {gap_A:.6f}')
print(f'Spectral gap B (2015-2025): {gap_B:.6f}')
print(f'Mean eigenvalue shift (B - A): {mean_shift:+.6f}')

## 6. Visualise Spectral Density (KDE Overlay)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- KDE overlay ---
x_grid = np.linspace(0, 2, 500)
kde_A = gaussian_kde(eigs_A, bw_method='scott')
kde_B = gaussian_kde(eigs_B, bw_method='scott')

ax = axes[0]
ax.fill_between(x_grid, kde_A(x_grid), alpha=0.35, color='steelblue', label='Period A (1999-2014)')
ax.fill_between(x_grid, kde_B(x_grid), alpha=0.35, color='tomato',   label='Period B (2015-2025)')
ax.plot(x_grid, kde_A(x_grid), color='steelblue', lw=1.5)
ax.plot(x_grid, kde_B(x_grid), color='tomato',   lw=1.5)
ax.set_title('Spectral Density — KDE Overlay')
ax.set_xlabel('Eigenvalue λ')
ax.set_ylabel('Density')
ax.legend()
ax.annotate(f'W₁ = {w1:.4f}', xy=(0.65, 0.88), xycoords='axes fraction', fontsize=10)

# --- Sorted eigenvalue rank plot ---
ax2 = axes[1]
ax2.plot(eigs_A, color='steelblue', lw=1.5, label='Period A')
ax2.plot(eigs_B, color='tomato',   lw=1.5, label='Period B')
ax2.set_title('Sorted Eigenvalues (rank plot)')
ax2.set_xlabel('Rank')
ax2.set_ylabel('Eigenvalue λ')
ax2.legend()

plt.suptitle('CVE Corpus — Graph Laplacian Spectral Drift', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/results/spectral_drift_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/results/spectral_drift_overview.png')

## 7. Interpretation Guide

| Metric | What it measures | Expected direction |
|--------|------------------|--------------------|
| Wasserstein-1 (W₁) | Earth-mover distance between spectral distributions | Higher = more drift |
| Spectral gap Δ | Connectivity / cluster separation of the kNN graph | Lower in B → denser, less modular community structure |
| Mean eigenvalue shift | Average shift of the whole spectrum | Positive → B is more connected on average |

A large W₁ together with a shrinking spectral gap in Period B
suggests that modern CVEs form **denser semantic clusters** —
i.e., vulnerability language has become more specialised and topic-concentrated
(e.g. cloud-native, supply-chain, ML/AI categories emerging post-2015).